In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def load_pairs(dataset_dir):
    """Returns list of (subject_id, path_img0, path_img1)."""
    files = glob.glob(os.path.join(dataset_dir, "*_0.jpg"))
    pairs = []
    for f in sorted(files):
        base = os.path.basename(f)
        subj_id = base.split("_0.jpg")[0]
        img1_path = os.path.join(dataset_dir, f"{subj_id}_1.jpg")
        if os.path.exists(img1_path):
            pairs.append((subj_id, f, img1_path))
    if not pairs:
        raise RuntimeError(f"No valid pairs found in {dataset_dir}")
    print(f"[INFO] Loaded {len(pairs)} subject pairs.")
    return pairs

In [ ]:
def add_gaussian_noise(img, mean=0, sigma=25):
    noise = np.random.normal(mean, sigma, img.shape).astype(np.float32)
    noisy = img.astype(np.float32) + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)


def add_salt_pepper_noise(img, amount=0.02, salt_vs_pepper=0.5):
    noisy = img.copy()
    h, w = img.shape[:2]
    num_salt = int(amount * h * w * salt_vs_pepper)
    num_pepper = int(amount * h * w * (1 - salt_vs_pepper))

    ys = np.random.randint(0, h, num_salt)
    xs = np.random.randint(0, w, num_salt)
    noisy[ys, xs] = 255

    ys = np.random.randint(0, h, num_pepper)
    xs = np.random.randint(0, w, num_pepper)
    noisy[ys, xs] = 0
    return noisy


def synthetic_resize(img, downscale_factor=0.25):
    """Simulate a low-res / synthetic capture: shrink then upscale back."""
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w * downscale_factor)),
                              max(1, int(h * downscale_factor))),
                        interpolation=cv2.INTER_LINEAR)
    restored = cv2.resize(small, (w, h), interpolation=cv2.INTER_CUBIC)
    return restored


CONDITIONS = {
    "Real_Image":        lambda img: img,
    "Gaussian_Noise":    lambda img: add_gaussian_noise(img),
    "SaltPepper_Noise":  lambda img: add_salt_pepper_noise(img),
    "Synthetic_Resize":  lambda img: synthetic_resize(img),
}

In [ ]:
def build_models():
    mtcnn = MTCNN(image_size=160, margin=20, post_process=True, device=DEVICE)
    facenet = InceptionResnetV1(pretrained='vggface2').eval().to(DEVICE)
    return mtcnn, facenet


def get_embedding(img_bgr, mtcnn, facenet):
    """img_bgr: numpy array (OpenCV BGR). Returns 512-d embedding or None if no face found."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    face_tensor = mtcnn(img_rgb)  # aligned+cropped face tensor, or None
    if face_tensor is None:
        return None
    face_tensor = face_tensor.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = facenet(face_tensor)
    return emb.squeeze(0).cpu().numpy()

In [ ]:
def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))



def compute_scores_for_condition(pairs, mtcnn, facenet, condition_fn, condition_name):
    embeddings_0, embeddings_1, valid_ids = [], [], []

    for subj_id, p0, p1 in pairs:
        img0 = cv2.imread(p0)
        img1 = cv2.imread(p1)
        if img0 is None or img1 is None:
            continue

        img0 = condition_fn(img0)
        img1 = condition_fn(img1)

        e0 = get_embedding(img0, mtcnn, facenet)
        e1 = get_embedding(img1, mtcnn, facenet)
        if e0 is None or e1 is None:
            print(f"[WARN] No face detected for subject {subj_id} under {condition_name}, skipping.")
            continue

        embeddings_0.append(e0)
        embeddings_1.append(e1)
        valid_ids.append(subj_id)

    n = len(valid_ids)
    genuine_scores, imposter_scores = [], []

    
    for i in range(n):
        genuine_scores.append(cosine_similarity(embeddings_0[i], embeddings_1[i]))

    
    for i in range(n):
        for j in range(n):
            if i != j:
                imposter_scores.append(cosine_similarity(embeddings_0[i], embeddings_1[j]))

    return np.array(genuine_scores), np.array(imposter_scores), valid_ids

In [ ]:

def compute_eer(genuine_scores, imposter_scores):
    labels = np.concatenate([np.ones(len(genuine_scores)), np.zeros(len(imposter_scores))])
    scores = np.concatenate([genuine_scores, imposter_scores])

    fpr, tpr, thresholds = roc_curve(labels, scores)
    fnr = 1 - tpr

    # EER is where FPR == FNR
    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    eer_threshold = thresholds[idx]
    roc_auc = auc(fpr, tpr)
    return eer, eer_threshold, fpr, tpr, roc_auc


def accuracy_at_threshold(genuine_scores, imposter_scores, threshold):
    tar = np.mean(genuine_scores >= threshold)   # True Accept Rate
    frr = 1 - tar                                # False Reject Rate
    far = np.mean(imposter_scores >= threshold)  # False Accept Rate
    return tar, frr, far

In [ ]:

def main(dataset_dir, outdir):
    os.makedirs(outdir, exist_ok=True)
    pairs = load_pairs(dataset_dir)
    mtcnn, facenet = build_models()

    results_table = []
    plt.figure(figsize=(7, 6))

    for cond_name, cond_fn in CONDITIONS.items():
        print(f"\n[INFO] Processing condition: {cond_name}")
        genuine, imposter, valid_ids = compute_scores_for_condition(
            pairs, mtcnn, facenet, cond_fn, cond_name
        )

        if len(genuine) == 0 or len(imposter) == 0:
            print(f"[WARN] Skipping {cond_name}: not enough valid data.")
            continue

        eer, eer_thr, fpr, tpr, roc_auc = compute_eer(genuine, imposter)
        tar, frr, far = accuracy_at_threshold(genuine, imposter, eer_thr)

        results_table.append({
            "Condition": cond_name,
            "Num_Subjects": len(valid_ids),
            "Mean_Genuine_Score": round(np.mean(genuine), 4),
            "Mean_Imposter_Score": round(np.mean(imposter), 4),
            "EER": round(eer * 100, 2),          # in %
            "EER_Threshold": round(eer_thr, 4),
            "TAR@EER_thr(%)": round(tar * 100, 2),
            "FAR@EER_thr(%)": round(far * 100, 2),
            "AUC": round(roc_auc, 4),
        })

        # Save score distributions
        plt.figure(figsize=(6, 4))
        plt.hist(genuine, bins=30, alpha=0.6, label="Genuine", color="green")
        plt.hist(imposter, bins=30, alpha=0.6, label="Imposter", color="red")
        plt.title(f"Genuine vs Imposter Scores — {cond_name}")
        plt.xlabel("Cosine Similarity Score")
        plt.ylabel("Frequency")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(outdir, f"dist_{cond_name}.png"))
        plt.close()

        # Add to combined ROC plot
        plt.figure(1)
        plt.plot(fpr, tpr, label=f"{cond_name} (AUC={roc_auc:.3f})")

    # Finalize combined ROC curve
    plt.figure(1)
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    plt.xlabel("False Positive Rate (FAR)")
    plt.ylabel("True Positive Rate (1 - FRR)")
    plt.title("ROC Curves — FaceNet Verification Across Conditions")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "roc_all_conditions.png"))
    plt.close()

    df = pd.DataFrame(results_table)
    df.to_csv(os.path.join(outdir, "results_summary.csv"), index=False)
    print("\n[RESULTS TABLE]")
    print(df.to_string(index=False))
    print(f"\n[INFO] All plots + CSV saved to: {outdir}")



DATASET_DIR = "./DataSet"   # <-- change if your DataSet folder is elsewhere
OUTDIR = "./results"        # <-- output folder for plots + CSV

main(DATASET_DIR, OUTDIR)

[INFO] Loaded 101 subject pairs.


  0%|          | 0.00/107M [00:00<?, ?B/s]


[INFO] Processing condition: Real_Image

[INFO] Processing condition: Gaussian_Noise

[INFO] Processing condition: SaltPepper_Noise

[INFO] Processing condition: Synthetic_Resize

[RESULTS TABLE]
       Condition  Num_Subjects  Mean_Genuine_Score  Mean_Imposter_Score  EER  EER_Threshold  TAR@EER_thr(%)  FAR@EER_thr(%)    AUC
      Real_Image           101              0.7392               0.0314 0.94         0.3943           99.01            0.89 0.9998
  Gaussian_Noise           101              0.6750               0.0694 2.99         0.3615           97.03            3.00 0.9958
SaltPepper_Noise           101              0.6648               0.0449 1.71         0.3842           98.02            1.45 0.9981
Synthetic_Resize           101              0.6168               0.0453 4.73         0.3056           95.05            4.51 0.9949

[INFO] All plots + CSV saved to: ./results
